# Evaluating the Warehouse Agent with Strands Evals & AgentCore Evaluations

In Labs 7 and 8 we deployed a warehouse operations agent to Amazon Bedrock AgentCore Runtime and gave it memory. But a correct-*looking* answer isn't proof the agent is good — before trusting it for real procurement decisions we need to **measure its quality systematically**.

This lab builds an evaluation suite and grades the agent in **two stages** — locally with [Strands Evals](https://pypi.org/project/strands-agents-evals/) first (the fast, cheap pre-deploy gate), then against the deployed AgentCore runtime with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/) reading its production OTEL traces. The grader design follows Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

> **This notebook runs standalone.** The setup section below re-initializes everything it needs (model, local agent, deployment details) — you don't need Lab 8 loaded in the same kernel. Stage 1 (local) needs only your SAP GenAI Hub credentials; Stage 2 (deployed) additionally needs the runtime deployed in Lab 7 and memory-enhanced in Lab 8.

Note: If you have not set up your ai-core credentials yet, please follow notebook 00-load-sap-ai-core-credentials.


## Prerequisites

1. **Completed Lab 7** — a warehouse agent deployed to AgentCore Runtime (writes `lab7_deployment.json`). Required for Stage 2.
2. **Completed Lab 8** (recommended) — so the deployed runtime is memory-enhanced. Stage 2 grades whatever is currently deployed.
3. SAP AI Core credentials in `~/.aicore/config.json` (run Lab 00 if not).
4. SAP S/4HANA Public Cloud API key (in `.env` or entered when prompted).
5. AWS credentials with AgentCore Evaluation permissions.

Stage 1 (local) runs with just items 1, 3, and 4. Stage 2 (deployed) additionally needs the deployed runtime and AWS permissions.


## 1. Setup — imports, configuration, and the local agent

This lab is self-contained: the next few cells import dependencies, validate your credentials, initialize the `SAPGenAIHubModel`, load the deployed-agent details from `lab7_deployment.json`, and define the local warehouse agent used for Stage 1. None of this depends on Lab 8 having run in the same kernel.


In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from util.odata_tool import odata_caller
from strands import Agent, tool

import os
import json
import time
import uuid
from datetime import datetime, timedelta
from collections import defaultdict

import boto3
from dotenv import load_dotenv
import getpass

# Load environment variables from .env file
load_dotenv()

# Prompt for the SAP API key if it isn't already set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")


In [ ]:
# Validate required configuration before proceeding
import os

_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json — run notebook 00 first")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set — check your .env file")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
# TODO: Choose your model — options: "anthropic--claude-4.5-sonnet", "amazon--nova-lite", "amazon--nova-pro"
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    max_tokens=4096,
)

# Load the deployed-agent details written by Lab 7 (lab7_deployment.json). Stage 2 evaluates
# THIS deployed runtime. Stage 1 (local) only needs `model`, but we load the deployment up
# front so the whole lab is configured in one place.
DEPLOYMENT_FILE = "lab7_deployment.json"
if not os.path.exists(DEPLOYMENT_FILE):
    raise SystemExit(
        f"{DEPLOYMENT_FILE} not found — run Lab 7 "
        "(07-deploy-warehouse-agent-to-agentcore.ipynb) first (and Lab 8 to add memory)."
    )

with open(DEPLOYMENT_FILE, "r") as f:
    deployment = json.load(f)

AGENT_NAME = deployment["agent_name"]
AGENT_ID = deployment["agent_id"]
AGENT_ARN = deployment["agent_arn"]
REGION = deployment.get("region") or "us-east-1"

print(f"Region: {REGION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"Model: {model.get_config()['model_id']} via SAP GenAI Hub")


### The local agent

Stage 1 grades a **local** instance of the warehouse agent — same model, system prompt, and tools as the deployed one, but run in-process so we can read exact tool-call counts straight off the Strands result. We run it **memory-free**: memory is a Lab 8 concern, and grading the agent's own query logic (not its recall) is what the local stage is for. The deployed, memory-enabled runtime is graded in Stage 2.


In [ ]:
# TODO: Customize the system prompt for your warehouse/domain
WAREHOUSE_SYSTEM_PROMPT = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750). 
You have access to real-time SAP warehouse data through OData APIs.

CORE CAPABILITIES:
1. Product Discovery: Find available products and their stock levels
2. Dynamic Querying: Construct intelligent OData queries based on user needs
3. Order Fulfillment: Check if orders can be fulfilled based on current inventory

AVAILABLE PRODUCTS:
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Provide specific, actionable insights with quantitative data
- If you know the user's preferences from memory, apply them without asking
- Do not use emojis

When users ask questions:
1. Determine what data you need
2. Use the odata_caller tool to query SAP S/4HANA APIs
3. Analyze results and provide comprehensive responses

For OData calls, use:
- base_url: "https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata4/sap/api_whse_physstockprod/srvd_a2x/sap/whsephysicalstockproducts/0001"
- auth_type: "api_key"
- auth_env_var: "SAP_S4HANA_PUBLIC_CLOUD_KEY"
"""

# TODO: Adjust capacity for your warehouse (used in evaluation scenarios)
WAREHOUSE_CAPACITY = 500


def create_warehouse_agent(with_memory=False):
    """Create the warehouse agent for LOCAL evaluation (memory-free).

    The `with_memory` parameter is kept for compatibility with the Stage 1 grader
    cells, but local evaluation always runs without memory: we grade the agent's
    query logic against the same model/prompt/tools as the deployed runtime, not its
    recall. The memory-enabled agent is the deployed one, graded in Stage 2.
    """
    return Agent(
        model=model,
        system_prompt=WAREHOUSE_SYSTEM_PROMPT,
        tools=[odata_caller],
    )


# Smoke-test that the local agent constructs.
_ = create_warehouse_agent()
print("Local warehouse agent ready for Stage 1 evaluation.")


## How we evaluate: two stages

A correct-looking answer isn't proof the agent is good. To measure quality we need to score the
agent's behaviour systematically — and we do it in **two stages**, local first, then
against the deployed runtime:

1. **Local evaluation with [Strands Evals](https://pypi.org/project/strands-agents-evals/)** —
   run the agent in this notebook and grade the results in-process. Fast, cheap, and needs no
   deployment, so it's the natural **pre-deploy gate**: you can iterate on the prompt or tools and
   re-grade in seconds. We use it here for both *code-based* and *LLM-as-judge* graders.
2. **Deployed evaluation with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/)** —
   score the memory-enhanced agent deployed in Lab 8 by reading its **production OTEL traces** from CloudWatch.
   This is what you run against the *real* runtime (right model, right infra, right latency) and
   what you'd wire into continuous production monitoring.

Same agent, graded twice — locally as you build, then in production as it runs.

```
  STAGE 1 — Local (this notebook)                STAGE 2 — Deployed (AgentCore Runtime)
  ┌───────────────────────────────┐             ┌────────────────────────────────────────┐
  │ create_warehouse_agent()      │             │ invoke_agent_runtime() ── OTEL ─► CW logs │
  │        │ run scenario         │             │        │                                  │
  │        ▼                      │             │        ▼   EvaluationClient.run()          │
  │ Strands Evals                 │             │ AgentCore built-in + custom evaluators    │
  │  • ToolCallBudget (code)      │             │  • Correctness / Helpfulness (LLM-judge)  │
  │  • ToolCalled (code)          │             │  • GoalSuccessRate / ToolSelectionAccuracy│
  │  • TrajectoryEvaluator (LLM)  │             │  • WarehouseOperationalQuality (custom)   │
  │  • ToolSelectionAccuracy (LLM)│             │                                            │
  └───────────────────────────────┘             └────────────────────────────────────────┘
     fast · cheap · pre-deploy gate                 real runtime · production traces
```


## Understanding grader types

Before we score anything, it helps to know *what kinds of graders exist*. Anthropic's
[Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)
groups them into three families — most real agent evals **combine** them, each grading some part of
the transcript or the outcome:

| Grader family | How it works | Strengths | Weaknesses | In this lab |
|---|---|---|---|---|
| **Code-based** | String match, binary tests, counting, static analysis — deterministic checks | Fast, cheap, objective, reproducible | Brittle to valid variations; lacks nuance | `ToolCallBudget`, `ToolCalled` (Strands Evals, local) |
| **Model-based (LLM-as-judge)** | An LLM scores against a rubric — rubric scoring, pairwise comparison, multi-judge consensus | Flexible, scalable, captures nuance | Non-deterministic, costs tokens, needs calibration against humans | `TrajectoryEvaluator`, `ToolSelectionAccuracyEvaluator` (local); `Correctness`, `Helpfulness`, `GoalSuccessRate`, `ToolSelectionAccuracy`, custom `WarehouseOperationalQuality` (deployed) |
| **Human** | SME review, spot-check sampling, inter-annotator agreement | Gold-standard quality | Expensive, slow | Not demoed — used to *calibrate* the LLM judges (read the transcripts!) |

The guidance: **use code-based graders where possible** (they're the cheapest and most reliable),
**LLM judges where you need flexibility or nuance**, and **humans judiciously** to validate and
calibrate the LLM judges.

### Outcome vs. trajectory

A second axis cuts across all three families — *what* you grade:

- **Outcome** — the final state or answer. *Did the agent get it right?* Built-in `Correctness`
  and our custom `WarehouseOperationalQuality` grade the outcome.
- **Trajectory** — the path the agent took to get there. *Did it use the right tools, efficiently?*
  `ToolCallBudget`, `ToolCalled`, `TrajectoryEvaluator`, and `ToolSelectionAccuracy` grade the trajectory.

The blog's advice is to lean on **outcome** grading where you can (checking a rigid tool-call
sequence is often too brittle) — **but** trajectory matters when *how* the agent works is itself
the thing you care about. Our `low-stock-items` scenario is exactly that case: the answer is
correct (outcome ✓) but reached via many redundant OData calls (trajectory ✗). Outcome-only
grading is structurally blind to it, which is why we add a code-based tool-call budget.


## Evaluation scenarios

These scenarios are **shared by both stages** — we grade the same set locally (Stage 1) and on the
deployed runtime (Stage 2). Each scenario contains:

- **prompt**: The user query to send to the agent
- **expected_response**: Description of what a good response looks like (used by Correctness + custom evaluator)
- **expected_trajectory**: Expected tool calls (for ToolSelectionAccuracy / TrajectoryEvaluator)
- **assertions**: Conditions for GoalSuccessRate
- **max_tool_calls**: Tool-call budget for the code-based efficiency grader (Stage 1)

The last scenario, `low-stock-items`, is drawn as an example of inefficient trajectory: the query "What
are the items with low stock?" used to trigger many redundant OData calls (repeated `$metadata`
discovery, trial-and-error `$filter` guessing) while still producing a correct answer. Outcome
graders like `Correctness`/`GoalSuccessRate` score that path 1.0 — they measure *whether* the
answer is right, not *how efficiently* it was reached. We attach a tool-call budget so the
code-based efficiency grader in Stage 1 can catch the wasteful trajectory.


In [ ]:
evaluation_scenarios = [
    {
        "name": "inventory-check-single",
        "prompt": "What is the current stock level for WM-AN02 Control Units?",
        "expected_response": (
            "The response should contain the specific stock quantity for WM-AN02 Control Units "
            "retrieved from the SAP warehouse API, with the product correctly identified as "
            "Control Units. The number should come from actual API data, not be invented."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent correctly identified WM-AN02 as Control Units. "
            "Agent reported a specific numeric stock quantity from the API."
        ),
        # Tool-call budget: one schema-discovery call + one data call is plenty.
        "max_tool_calls": 2,
    },
    {
        "name": "inventory-overview",
        "prompt": "Give me a complete overview of all products currently in the warehouse.",
        "expected_response": (
            "The response should list all warehouse products (WM-AN01 Advanced Sensors, "
            "WM-AN02 Control Units, WM-AN03 Power Modules, WM-AN04 Communication Devices) "
            "with their current stock quantities from the API, presented in a structured format."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via OData. "
            "Agent listed multiple products with stock quantities. "
            "Agent presented results in a structured, readable format."
        ),
        # One unfiltered query returns all products; a second is the most that should be needed.
        "max_tool_calls": 2,
    },
    {
        "name": "fulfillment-feasibility",
        "prompt": "Can we fulfill an order for 200 units of WM-AN02 Control Units?",
        "expected_response": (
            "The response should check current WM-AN02 stock from the API, compare it against "
            "the requested 200 units, and provide a clear yes/no fulfillment recommendation "
            "with the actual available quantity."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried current WM-AN02 stock via OData. "
            "Agent compared available quantity against the 200-unit request. "
            "Agent provided a clear yes/no fulfillment answer with supporting data."
        ),
        "max_tool_calls": 2,
    },
    {
        # Regression scenario drawn from a real workshop failure. This query used to trigger
        # many redundant OData calls (repeated $metadata discovery, trial-and-error $filter
        # guessing). Correctness/GoalSuccessRate scored it 1.0 despite the waste, so we pair it
        # with the tool-call-efficiency grader (Stage 1) to catch the inefficient trajectory.
        "name": "low-stock-items",
        "prompt": "What are the items with low stock?",
        "expected_response": (
            "The response should identify which products are low on stock (or state that none "
            "are below the reorder threshold), based on stock quantities from the SAP warehouse "
            "API. The agent should reach the answer efficiently — ideally a single filtered or "
            "sorted OData query — rather than fetching everything and repeatedly rediscovering the schema."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent determined which products are low on stock relative to their capacity. "
            "Agent reached the answer without redundant, repeated OData calls."
        ),
        # Same tight budget: a low-stock check should be one filtered/sorted query.
        "max_tool_calls": 2,
    },
]

print(f"Evaluation scenarios defined: {len(evaluation_scenarios)}")
for s in evaluation_scenarios:
    print(f"  - {s['name']}: {s['prompt']} (budget: {s['max_tool_calls']} tool calls)")

## Stage 1 — Local evaluation with Strands Evals

We start with local evaluation because it is cost-efficient and fastest feedback loop: run the agent right here in the
notebook, grade it in-process, iterate. No deployment, no waiting for traces to land in CloudWatch.

**[Strands Evals](https://pypi.org/project/strands-agents-evals/)** (`strands-agents-evals`) is the
evaluation framework that ships with the Strands ecosystem — already a dependency of this project.
It gives us evaluators from **both** grader families in the taxonomy above:

- **Code-based (deterministic, no LLM):**
  - **`ToolCalled`** — checks a given tool was used at all.
  - **`ToolCallBudget`** — our own `Evaluator` subclass that counts tool calls against a
    per-scenario budget. This is the blog's canonical **code-based grader**: fast, cheap,
    objective, reproducible.
- **Model-based (LLM-as-judge):**
  - **`TrajectoryEvaluator`** — scores the tool-call trajectory against a natural-language rubric.
  - **`ToolSelectionAccuracyEvaluator`** — judges whether each tool call was justified.

  Both route through the **same `SAPGenAIHubModel`** the agent uses — so the judge runs on SAP
  GenAI Hub too, no separate Bedrock access required.

**Where the trajectory comes from.** We run each scenario against a **local** agent
(`create_warehouse_agent(with_memory=False)`) and read the tool-call counts straight off the
Strands SDK result:

```python
result = agent("What are the items with low stock?")
result.metrics.tool_metrics["odata_caller"].call_count   # exact count, no span parsing
```

That's exact and needs no log parsing or ingestion wait. The trade-off: we're grading the *local*
agent (same model, prompt, and tools as the deployed one), not the exact deployed runtime — which
is fine for a pre-deploy gate. Stage 2 grades the deployed runtime from its real traces.

A scenario can **pass correctness but fail the tool-call budget** — that's the signal we want. The
`low-stock-items` scenario (a real workshop failure) does exactly this: correct answer, wasteful
path. Reading the flagged trajectory then points at concrete fixes (put field names and a working
`$filter` example in the system prompt, or give `odata_caller` a schema hint) that an
outcome-only "Correctness: 1.0" score would never surface.


In [ ]:
# Code-based graders with Strands Evals — deterministic, no LLM.
from strands_evals.evaluators import ToolCalled, Evaluator
from strands_evals.types import EvaluationData, EvaluationOutput

# Name of the tool we count. TODO: change if your agent uses a different tool.
TOOL_NAME = "odata_caller"


class ToolCallBudget(Evaluator):
    """Deterministic grader: fail if a tool is called more than `max_calls` times.

    Subclasses the Strands Evals `Evaluator` so it composes with the framework's
    types and reporting. No LLM — it just counts entries in the trajectory list.
    """

    def __init__(self, tool_name: str, max_calls: int):
        super().__init__()
        self.tool_name = tool_name
        self.max_calls = max_calls

    def evaluate(self, case: EvaluationData) -> list[EvaluationOutput]:
        trajectory = case.actual_trajectory or []
        n = sum(1 for t in trajectory if t == self.tool_name)
        within = n <= self.max_calls
        return [
            EvaluationOutput(
                score=1.0 if within else 0.0,
                test_pass=within,
                reason=f"{n} '{self.tool_name}' call(s) vs budget of {self.max_calls}",
                label="efficient" if within else "inefficient",
            )
        ]


def run_local_agent_trajectory(prompt: str):
    """Run the scenario against a LOCAL agent and return (output_text, tool_trajectory).

    The Strands SDK exposes exact per-tool counts on the result, so we expand them
    into a flat list of tool names (one entry per call) that the evaluators consume.
    We use a memory-free agent so tool calls reflect this query alone.
    """
    agent = create_warehouse_agent(with_memory=False)
    result = agent(prompt)
    output_text = str(result.message)

    trajectory = []
    tool_metrics = getattr(result.metrics, "tool_metrics", {}) or {}
    for tool_name, m in tool_metrics.items():
        trajectory.extend([tool_name] * getattr(m, "call_count", 0))
    return output_text, trajectory


# Run each scenario locally once and cache (output, trajectory, case) for reuse by the
# LLM-judge cell below — so we invoke the agent once per scenario, not once per grader.
local_runs = {}
efficiency_results = {}

print("Running Strands Evals code-based graders (local agent runs)...\n")

for scenario in evaluation_scenarios:
    scenario_name = scenario["name"]
    max_calls = scenario["max_tool_calls"]

    output_text, trajectory = run_local_agent_trajectory(scenario["prompt"])

    # Assemble the Strands Evals case once, reuse across evaluators.
    case = EvaluationData(
        name=scenario_name,
        input=scenario["prompt"],
        actual_output=output_text,
        actual_trajectory=trajectory,
        expected_trajectory=scenario["expected_trajectory"],
    )
    local_runs[scenario_name] = {"case": case, "trajectory": trajectory, "output": output_text}

    tool_called = ToolCalled(TOOL_NAME).evaluate(case)[0]
    budget = ToolCallBudget(TOOL_NAME, max_calls).evaluate(case)[0]

    n_calls = sum(1 for t in trajectory if t == TOOL_NAME)
    # Shape the result like the AgentCore evaluator dicts so it slots into the summary table.
    efficiency_results[scenario_name] = [{
        "evaluatorId": "StrandsEvals.ToolCallBudget",
        "value": budget.score,
        "label": budget.label,
        "explanation": budget.reason,
        "n_tool_calls": n_calls,
        "max_tool_calls": max_calls,
        "tool_called": tool_called.test_pass,
    }]

    print(f"  {scenario_name}: budget={budget.score} ({budget.label}); "
          f"tool_called={tool_called.test_pass}")
    print(f"    {budget.reason}\n")

print("Code-based grading complete.")


### LLM-as-judge graders (local)

The tool-call budget is deterministic — it can't tell you *why* a trajectory is good or bad, only
whether it exceeded a number. For nuanced trajectory judgement we bring in Strands Evals'
**LLM-as-judge** evaluators, both routed through the **same `SAPGenAIHubModel`** the agent uses:

- **`TrajectoryEvaluator`** — scores the tool-call trajectory against a natural-language **rubric**.
  It works on the flat trajectory list we already captured, so no re-run needed.
- **`ToolSelectionAccuracyEvaluator`** — a **tool-level** judge that asks, for each call, "was this
  tool call justified here?" It needs richer context than a flat list (a Strands `Session` built
  from OTEL spans), so we drive it through Strands Evals' `Experiment` + `TracedHandler` harness,
  which captures spans from a fresh local run.

> These make **live LLM calls** through SAP GenAI Hub, so this cell is slower than the deterministic
> graders and its scores will vary slightly between runs (that's the nature of model-based grading —
> the blog calls this out and recommends calibrating judges against human labels). The
> `ToolSelectionAccuracyEvaluator` path re-runs the agent to capture traces; if local span capture
> isn't available in your environment it degrades gracefully and the `TrajectoryEvaluator` result
> still stands.


In [ ]:
# LLM-as-judge graders with Strands Evals — routed through the same SAPGenAIHubModel.
from strands_evals.evaluators import TrajectoryEvaluator, ToolSelectionAccuracyEvaluator

# Rubric for the trajectory judge. Efficiency is the thing we care about for these scenarios.
TRAJECTORY_RUBRIC = (
    "Score the tool-call trajectory for a warehouse inventory agent. A good trajectory answers the "
    "user's question with as few tool calls as possible — ideally one filtered/sorted OData query. "
    "Penalize redundant calls such as re-discovering the schema or trial-and-error $filter guessing. "
    "Score 1.0 for an efficient, well-chosen trajectory; lower toward 0.0 as redundant calls increase."
)

llm_judge_results = {}

print("Running Strands Evals LLM-as-judge graders (via SAP GenAI Hub)...\n")

# --- TrajectoryEvaluator: works on the flat trajectory we already captured locally. ---
trajectory_judge = TrajectoryEvaluator(rubric=TRAJECTORY_RUBRIC, model=model)

for scenario_name, run in local_runs.items():
    try:
        out = trajectory_judge.evaluate(run["case"])[0]
        llm_judge_results[scenario_name] = [{
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        }]
        print(f"  {scenario_name}: TrajectoryEvaluator={out.score:.2f} — {(out.reason or '')[:100]}")
    except Exception as e:
        print(f"  {scenario_name}: TrajectoryEvaluator ERROR: {e}")
        llm_judge_results[scenario_name] = []

# --- ToolSelectionAccuracyEvaluator: tool-level judge that needs a Session (OTEL spans). ---
# We use the Strands Evals Experiment + TracedHandler harness, which runs the agent and captures
# spans into a Session the tool-level judge can parse. This is heavier (re-runs the agent) and
# depends on local telemetry capture, so we guard it and fall back gracefully.
#
# We call `run_evaluations_async` and `await` it rather than the sync `run_evaluations`: the sync
# wrapper calls `asyncio.run()` internally, which raises inside a Jupyter kernel (it already has a
# running event loop). Notebook cells support top-level `await`, so the async variant is the right
# entry point here. `max_workers=1` keeps runs sequential (one SAP GenAI Hub call at a time),
# matching the sync wrapper's behaviour.
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    cases = [
        Case(
            name=s["name"],
            input=s["prompt"],
            expected_trajectory=s["expected_trajectory"],
        )
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def warehouse_eval_task():
        # Fresh memory-free agent per case; the handler captures its spans into a Session.
        return create_warehouse_agent(with_memory=False)

    experiment = Experiment(
        cases=cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    reports = await experiment.run_evaluations_async(warehouse_eval_task, max_workers=1)

    # Each EvaluationReport carries parallel lists: cases[i] (a dict) and scores[i].
    # Fold the tool-selection scores into llm_judge_results, keyed by scenario name.
    for report in reports:
        for case_dict, score in zip(report.cases, report.scores):
            name = case_dict.get("name") if isinstance(case_dict, dict) else None
            if name is not None and score is not None:
                llm_judge_results.setdefault(name, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {name}: ToolSelectionAccuracy={score:.2f}")
    print("\nLLM-as-judge grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracyEvaluator skipped (local trace capture unavailable): {e}")
    print("  TrajectoryEvaluator results above still stand.")


### Local results

We combine the code-based and LLM-judge scores into one local table. The key thing to look for is
a scenario that **reached an answer (the tool was called) but blew its tool-call budget** — a
correct-but-inefficient trajectory. That's the `low-stock-items` failure mode, and it's exactly
what an outcome-only grader would miss.


In [ ]:
# Combine local Strands Evals scores (code-based + LLM-judge) into one table.
local_results = {}
for scenario in evaluation_scenarios:
    name = scenario["name"]
    local_results[name] = (
        efficiency_results.get(name, [])
        + llm_judge_results.get(name, [])
    )

LOCAL_EVALUATOR_IDS = [
    "StrandsEvals.ToolCallBudget",
    "StrandsEvals.TrajectoryEvaluator",
    "StrandsEvals.ToolSelectionAccuracy",
]


def _local_short_name(eid):
    return {
        "StrandsEvals.ToolCallBudget": "ToolBudget",
        "StrandsEvals.TrajectoryEvaluator": "TrajJudge",
        "StrandsEvals.ToolSelectionAccuracy": "ToolSelect",
    }.get(eid, eid.split(".")[-1][:12])


print("=" * 78)
print(" LOCAL EVALUATION — STRANDS EVALS (code-based + LLM-judge)")
print("=" * 78)

header = f"{'Scenario':<25}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid):>14}"
print(header)
print("-" * 78)

local_scenario_scores = defaultdict(dict)
for name, results in local_results.items():
    row = f"{name:<25}"
    for eid in LOCAL_EVALUATOR_IDS:
        score = next((r.get("value", "-") for r in results if r.get("evaluatorId") == eid), "-")
        if isinstance(score, (int, float)):
            row += f" {score:>14.2f}"
            local_scenario_scores[name][eid] = score
        else:
            row += f" {str(score):>14}"
    print(row)
print("=" * 78)

# Highlight correct-but-inefficient: the tool was called (agent reached an answer) but the
# tool-call budget was exceeded. Locally we don't have Builtin.Correctness, so ToolCalled is
# our "reached an answer" proxy — the point is a right-looking answer via a wasteful path.
print("\nCorrect-but-inefficient check (tool used, but over tool-call budget):")
flagged = False
for name, results in local_results.items():
    eff = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.ToolCallBudget"), {})
    if eff.get("tool_called") and eff.get("value") == 0.0:
        flagged = True
        print(f"  [!] {name}: reached an answer, but {eff.get('explanation', '')}")
if not flagged:
    print("  None flagged — every answer was reached within its tool-call budget.")


---

## Stage 2 — Deployed evaluation with AgentCore

Local grading is the pre-deploy gate. Now we grade the memory-enhanced agent **deployed to AgentCore Runtime in Lab 8** — the real runtime, with production instrumentation. Instead of reading tool counts off an
in-process result, **AgentCore Evaluations** scores the agent from its **OpenTelemetry traces**:

1. The deployed agent emits OTEL spans to CloudWatch Logs (configured in Lab 7).
2. We invoke it for each scenario and wait (~180s) for spans to be ingested.
3. `EvaluationClient.run()` reads those spans and scores them with built-in or custom LLM-as-judge evaluators.

This is what you run against the *actual* production agent, and the same mechanism you'd wire into
continuous online evaluation for monitoring.

### Configure evaluation infrastructure

Set up the AgentCore Runtime client and helper functions for invoking the deployed agent and
waiting for OTEL span ingestion.


In [ ]:
from datetime import timedelta

# AgentCore Runtime client for invoking the deployed agent
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# Derive the CloudWatch log group where OTEL spans land
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

# Ingestion delay — time to wait for OTEL spans to arrive in CloudWatch.
INGESTION_DELAY = 180


def invoke_with_retry(client, agent_arn, session_id, prompt, max_retries=3, wait=30):
    """Invoke agent runtime with retry for cold start 500 errors."""
    for attempt in range(max_retries):
        try:
            response = client.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                runtimeSessionId=session_id,
                payload=json.dumps({"prompt": prompt}).encode("utf-8"),
            )
            return response
        except client.exceptions.RuntimeClientError as e:
            if attempt < max_retries - 1:
                print(f"  Runtime error (attempt {attempt + 1}/{max_retries}), retrying in {wait}s (likely cold start)...")
                time.sleep(wait)
                session_id = f"{session_id}_r{attempt + 1}"
            else:
                raise e
    return None


print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"CloudWatch Log Group: {CW_LOG_GROUP}")
print(f"Ingestion delay: {INGESTION_DELAY}s")

### Create the custom evaluator

We register a domain-specific LLM-as-a-judge evaluator in the AgentCore control plane. This complements the built-in evaluators with SAP warehouse-specific scoring criteria that generic evaluators cannot assess.

**WarehouseOperationalQuality** (TRACE-level): Evaluates whether the agent's response is operationally useful for a warehouse manager — does it provide actionable inventory insights, use correct product codes, and present data in a way that supports procurement decisions?

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

_SUFFIX = uuid.uuid4().hex[:8]

# TODO: Use the inference profile matching your region (us.* for us-east-1, eu.* for eu-central-1)
JUDGE_MODEL_ID = "us.amazon.nova-pro-v1:0"

# Custom TRACE-level evaluator: Warehouse Operational Quality
print("Creating WarehouseOperationalQuality evaluator (TRACE-level)...")
warehouse_quality_response = agentcore_control.create_evaluator(
    evaluatorName=f"WarehouseOperationalQuality_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "You are a warehouse operations expert evaluating an AI assistant that queries "
                "SAP S/4HANA warehouse APIs for inventory management.\n\n"
                "Conversation context: {context}\n"
                "Agent response: {assistant_turn}\n"
                "Expected behavior: {expected_response}\n\n"
                "Evaluate the OPERATIONAL QUALITY of the response for a warehouse manager. Score based on:\n"
                "1. Does the response contain specific, quantitative inventory data (not vague statements)?\n"
                "2. Are SAP product codes (WM-AN01, WM-AN02, WM-AN03, WM-AN04) used correctly?\n"
                "3. Is the data presented in a way that supports immediate operational decisions "
                "(e.g., reorder recommendations, fulfillment feasibility, capacity utilization)?\n"
                "4. Does the response avoid hallucinating inventory numbers when API data is unavailable?\n\n"
                "Important: If the agent successfully queried the API and returned real data with "
                "correct product codes and actionable insights, score 1.0 even if formatting differs "
                "from the expected response.\n\n"
                "You MUST respond with EXACTLY one of these scores:\n"
                "- 0.0 if the response lacks inventory data, hallucinates numbers, or is not actionable\n"
                "- 0.5 if the response has some useful data but is missing key operational context\n"
                "- 1.0 if the response provides accurate, actionable warehouse intelligence\n\n"
                "Respond with only the numeric score (0.0, 0.5, or 1.0) on the first line, "
                "followed by a one-sentence explanation on the next line."
            ),
            "ratingScale": {
                "numerical": [
                    {"value": 0.0, "label": "not_actionable", "definition": "Response lacks data, hallucinates numbers, or provides no operational value."},
                    {"value": 0.5, "label": "partially_useful", "definition": "Some useful data present but missing key operational context for decisions."},
                    {"value": 1.0, "label": "operationally_excellent", "definition": "Accurate, specific, and actionable warehouse intelligence."},
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": JUDGE_MODEL_ID,
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_EVALUATOR_ID = warehouse_quality_response["evaluatorId"]
print(f"  Created: {CUSTOM_EVALUATOR_ID}")
print(f"\nCustom evaluator registered in AgentCore control plane.")

### Invoke the deployed agent for each scenario

We invoke the deployed agent (from Lab 7) for each evaluation scenario. Each invocation gets a unique `runtimeSessionId` so the evaluator can locate its spans independently.

In [ ]:
# Invoke the deployed agent for each scenario and collect session IDs
eval_sessions = []

print("Invoking deployed agent for each evaluation scenario...\n")


def parse_agent_response(response_body: str) -> str:
    """Parse invoke_agent_runtime response into plain text.

    The response is a JSON object: {"role": "assistant", "content": [{"text": "..."}], "metadata": {...}}
    It may arrive as a single blob or as multiple newline-delimited chunks that concatenate into one object.
    """
    text_parts = []

    # Try parsing as a single JSON object (most common)
    try:
        data = json.loads(response_body)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Fall back: response may be multiple concatenated JSON chunks (chunked transfer)
    try:
        combined = "".join(response_body.strip().split("\n"))
        data = json.loads(combined)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Last resort: return raw truncated
    return response_body[:500]


for scenario in evaluation_scenarios:
    session_id = f"eval_{scenario['name']}_{uuid.uuid4().hex}"

    try:
        response = invoke_with_retry(
            agentcore_client, AGENT_ARN, session_id, scenario["prompt"]
        )

        response_body = response["response"].read().decode("utf-8")
        agent_text = parse_agent_response(response_body)

        eval_sessions.append({
            "scenario_name": scenario["name"],
            "session_id": session_id,
            "prompt": scenario["prompt"],
            "response": agent_text[:500],
            "expected_response": scenario["expected_response"],
            "expected_trajectory": scenario["expected_trajectory"],
            "assertions": scenario["assertions"],
            "max_tool_calls": scenario["max_tool_calls"],
        })

        print(f"  [{scenario['name']}] Session: {session_id}")
        print(f"    Response: {agent_text[:100]}...\n")

    except Exception as e:
        print(f"  [{scenario['name']}] ERROR: {e}\n")

print(f"\nCompleted {len(eval_sessions)} agent invocations.")
print(f"Waiting {INGESTION_DELAY}s for CloudWatch span ingestion...")
time.sleep(INGESTION_DELAY)
print("Spans should now be available for evaluation.")

### Evaluate with built-in evaluators

We use `EvaluationClient.run()` to score each session with AgentCore's built-in evaluators:

| Evaluator | Level | Needs Ground Truth | What it measures |
|-----------|-------|-------------------|-----------------|
| `Builtin.Correctness` | TRACE | `expected_response` | Factual accuracy of the response |
| `Builtin.Helpfulness` | TRACE | None | How useful/valuable the response is |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` | Whether the agent completed the user's goal |
| `Builtin.ToolSelectionAccuracy` | SESSION | None | Whether the agent chose the right tools |

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

ec = EvaluationClient(region_name=REGION)

# Pre-populate the evaluator level cache (required for the SDK to route correctly)
ec._evaluator_level_cache.update({
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.ToolSelectionAccuracy": "SESSION",
})

BUILTIN_EVALUATOR_IDS = [
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.GoalSuccessRate",
    "Builtin.ToolSelectionAccuracy",
]

builtin_results = {}

print("Running built-in evaluators on each session...\n")

for session in eval_sessions:
    print(f"  Evaluating: {session['scenario_name']} (session: {session['session_id']})")

    try:
        results = ec.run(
            evaluator_ids=BUILTIN_EVALUATOR_IDS,
            agent_id=AGENT_ID,
            session_id=session["session_id"],
            look_back_time=timedelta(hours=1),
            reference_inputs=ReferenceInputs(
                expected_response=session["expected_response"],
                expected_trajectory=session["expected_trajectory"],
                assertions=[session["assertions"]],
            ),
        )
        builtin_results[session["scenario_name"]] = results
        for r in results:
            score = r.get("value", "N/A")
            label = r.get("label", "")
            evaluator = r.get("evaluatorId", "unknown")
            print(f"    {evaluator}: {score} ({label})")
    except Exception as e:
        print(f"    ERROR: {e}")
        builtin_results[session["scenario_name"]] = []

    print()

print("Built-in evaluation complete.")

### Evaluate with the custom evaluator

Now we run our domain-specific **WarehouseOperationalQuality** evaluator on the same sessions. This TRACE-level evaluator scores whether the agent provides actionable warehouse intelligence — something the generic built-in evaluators cannot assess.

In [ ]:
# Add custom evaluator level to the cache
ec._evaluator_level_cache[CUSTOM_EVALUATOR_ID] = "TRACE"

custom_results = {}

print("Running WarehouseOperationalQuality evaluator on each session...\n")

for session in eval_sessions:
    print(f"  Evaluating: {session['scenario_name']} (session: {session['session_id']})")

    try:
        results = ec.run(
            evaluator_ids=[CUSTOM_EVALUATOR_ID],
            agent_id=AGENT_ID,
            session_id=session["session_id"],
            look_back_time=timedelta(hours=1),
            reference_inputs=ReferenceInputs(
                expected_response=session["expected_response"],
                expected_trajectory=session["expected_trajectory"],
                assertions=[session["assertions"]],
            ),
        )
        custom_results[session["scenario_name"]] = results
        for r in results:
            score = r.get("value", "N/A")
            label = r.get("label", "")
            explanation = r.get("explanation", "")
            print(f"    Score: {score} ({label})")
            if explanation:
                print(f"    Reason: {explanation[:120]}")
    except Exception as e:
        print(f"    ERROR: {e}")
        custom_results[session["scenario_name"]] = []

    print()

print("Custom evaluation complete.")

## Display deployed evaluation results

Aggregate the AgentCore built-in and custom evaluator scores (from the deployed runtime's OTEL
traces) into a summary table. We also tie back to Stage 1: a scenario can score `Correctness` 1.0
on the deployed runtime yet have been flagged over-budget by the local code-based grader — the
outcome-vs-trajectory gap in action.


In [ ]:
# Combine the deployed AgentCore scores (built-in + custom LLM judge) into one table.
deployed_results = {}
for scenario_name in builtin_results:
    deployed_results[scenario_name] = (
        builtin_results.get(scenario_name, [])
        + custom_results.get(scenario_name, [])
    )

deployed_evaluator_ids = BUILTIN_EVALUATOR_IDS + [CUSTOM_EVALUATOR_ID]


def _short_name(eid):
    if eid == CUSTOM_EVALUATOR_ID:
        return "OpsQuality"
    return eid.split(".")[-1][:14]


print("=" * 92)
print(" DEPLOYED EVALUATION — AGENTCORE (built-in + custom LLM judge, from OTEL traces)")
print("=" * 92)

# Header
header = f"{'Scenario':<25}"
for eid in deployed_evaluator_ids:
    header += f" {_short_name(eid):>14}"
print(header)
print("-" * 92)

# Per-scenario scores
scenario_scores = defaultdict(dict)
for scenario_name, results in deployed_results.items():
    row = f"{scenario_name:<25}"
    for eid in deployed_evaluator_ids:
        score = next(
            (r.get("value", "-") for r in results if r.get("evaluatorId") == eid),
            "-",
        )
        if isinstance(score, (int, float)):
            row += f" {score:>14.2f}"
            scenario_scores[scenario_name][eid] = score
        else:
            row += f" {str(score):>14}"
    print(row)

# Averages
print("-" * 92)
avg_row = f"{'AVERAGE':<25}"
for eid in deployed_evaluator_ids:
    scores = [
        scenario_scores[s][eid]
        for s in scenario_scores
        if eid in scenario_scores[s]
    ]
    if scores:
        avg_row += f" {sum(scores)/len(scores):>14.2f}"
    else:
        avg_row += f" {'-':>14}"
print(avg_row)
print("=" * 92)

# Tie the two stages together: a deployed answer can score Correctness 1.0 while the local
# code-based grader flagged the same scenario as over-budget — outcome right, trajectory wasteful.
print("\nCross-stage insight (deployed Correctness=1.0 but local ToolBudget flagged it):")
flagged = False
for scenario_name, scores in scenario_scores.items():
    local_eff = next(
        (r for r in local_results.get(scenario_name, [])
         if r.get("evaluatorId") == "StrandsEvals.ToolCallBudget"),
        {},
    )
    if scores.get("Builtin.Correctness") == 1.0 and local_eff.get("value") == 0.0:
        flagged = True
        print(f"  [!] {scenario_name}: correct on the deployed runtime, but locally {local_eff.get('explanation', '')}")
if not flagged:
    print("  None flagged — no scenario was both deployed-correct and locally over-budget.")

# Show detailed deployed results for one scenario
print(f"\nDetailed deployed results for '{eval_sessions[0]['scenario_name']}':")
for r in deployed_results.get(eval_sessions[0]["scenario_name"], []):
    print(f"  {r.get('evaluatorId', 'unknown')}:")
    print(f"    Score: {r.get('value', 'N/A')} | Label: {r.get('label', 'N/A')}")
    if r.get("explanation"):
        print(f"    Explanation: {r['explanation'][:200]}")


## Cleanup (optional)

In [ ]:
# Uncomment to clean up the custom evaluator created in this lab.
# The AgentCore Memory resource and the deployed runtime are owned by Labs 8 and 7 —
# clean those up from their respective notebooks.

# agentcore_control.delete_evaluator(evaluatorId=CUSTOM_EVALUATOR_ID)
# print(f"Deleted evaluator: {CUSTOM_EVALUATOR_ID}")


## Summary

You evaluated the warehouse agent in **two stages**, grounded in Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

**Grader taxonomy:** code-based (fast, objective, reproducible), model-based / LLM-as-judge (flexible, nuanced), and human (gold standard, for calibration) — plus the outcome-vs-trajectory distinction.

**Stage 1 — Local evaluation with Strands Evals** (fast, cheap pre-deploy gate):
- Code-based: `ToolCallBudget` + `ToolCalled`, fed by the SDK's exact `tool_metrics.call_count`
- LLM-as-judge: `TrajectoryEvaluator` and `ToolSelectionAccuracyEvaluator`, routed through the same `SAPGenAIHubModel`
- Caught **correct-but-inefficient** answers (the low-stock query) that outcome graders score 1.0

**Stage 2 — Deployed evaluation with AgentCore** (real runtime, from OTEL traces):
- Built-in evaluators (Correctness, Helpfulness, GoalSuccessRate, ToolSelectionAccuracy)
- A custom LLM-as-judge evaluator (WarehouseOperationalQuality) for domain scoring

**Key takeaway:** evaluate locally first, then in production; combine code-based and model-based graders so you measure not just *whether* the agent is right (outcome) but *how efficiently* it gets there (trajectory).

**Next steps (see the evaluation harness spec):** negative / out-of-scope scenarios, multi-trial runs with pass@k / pass^k, chaos/fault-injection (Strands Evals 1.0+), and a pass/fail regression gate.
